### Purpose: 

Analyse marketing channel performance - which channels are worth spending on and which are wasting money.

Reads data from `marts.fact_marketing`

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

BASE_DIR = Path.cwd().parent

# Read environment variables
db_path_env = os.getenv("DB_PATH")

if db_path_env is None:
    raise ValueError("DB_PATH not found in .env")

DB_PATH = BASE_DIR / db_path_env    

con = duckdb.connect(str(DB_PATH), read_only=True)

# Confirm marketing table exists and has rows
print(con.execute("SELECT COUNT(*) FROM marts.fact_marketing").df())

In [ ]:
# ROAS by channel summary
df_roas = con.execute("""
    SELECT
        channel,
        ROUND(SUM(revenue_attributed), 2)   AS total_revenue,
        ROUND(SUM(roas), 2)                 AS total_roas,
        COUNT(*)                            AS records
    FROM marts.fact_marketing
    GROUP BY channel
    ORDER BY total_roas DESC
""").df()

# Spend vs revenue per channel (scatter) - using revenue_attributed & roas
df_scatter = con.execute("""
    SELECT
        channel,
        ROUND(AVG(roas), 2)                 AS avg_roas,
        ROUND(SUM(revenue_attributed), 2)   AS total_revenue,
        ROUND(AVG(ctr), 4)                  AS avg_ctr,
        ROUND(AVG(cpa), 2)                  AS avg_cpa
    FROM marts.fact_marketing
    GROUP BY channel
    ORDER BY avg_roas DESC
""").df()

# CPA by channel
df_cpa = con.execute("""
    SELECT
        channel,
        ROUND(AVG(cpa), 2)                  AS avg_cpa,
        ROUND(MIN(cpa), 2)                  AS min_cpa,
        ROUND(MAX(cpa), 2)                  AS max_cpa,
        COUNT(*)                            AS campaigns
    FROM marts.fact_marketing
    GROUP BY channel
    ORDER BY avg_cpa ASC
""").df()

# Best and worst performing channels by ROAS rank
df_rank = con.execute("""
    SELECT
        channel,
        ROUND(SUM(revenue_attributed), 2)   AS total_revenue,
        ROUND(AVG(roas), 2)                 AS avg_roas,
        ROUND(AVG(cpa), 2)                  AS avg_cpa,
        ROUND(AVG(ctr), 4)                  AS avg_ctr,
        RANK() OVER (ORDER BY AVG(roas) DESC) AS roas_rank
    FROM marts.fact_marketing
    GROUP BY channel
    ORDER BY roas_rank
""").df()


### Chart - ROAS by channel over time

In [ ]:
df = con.execute("""
    SELECT campaign_month, channel, roas
    FROM marts.fact_marketing
    ORDER BY campaign_month
""").df()

df['campaign_month'] = pd.to_datetime(df['campaign_month'])

fig, ax = plt.subplots(figsize=(13, 5))
for channel in df['channel'].unique():
    sub = df[df['channel'] == channel]
    ax.plot(sub['campaign_month'], sub['roas'], marker='o', label=channel, linewidth=2)

ax.axhline(1.0, color='red', linestyle='--', linewidth=0.8, label='Break-even (ROAS = 1.0)')
ax.set_title('ROAS by channel over time', fontsize=14, pad=12)
ax.set_xlabel('Month')
ax.set_ylabel('ROAS (revenue / spend)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(
    output_dir / "roas_trend.png",
    dpi=150,
    bbox_inches='tight'
)
plt.show()

`Organic and Email consistently deliver the highest ROAS — both are low-cost channels where investment compounds over time. Instagram and Facebook show higher volatility, suggesting campaign-level execution quality drives results more than channel-level allocation.`

### Chart - Spend vs revenue scatter (bubble = conversion count)

In [ ]:
df = con.execute("""
    SELECT
        channel,
        SUM(spend) AS total_spend,
        SUM(revenue_attributed) AS total_revenue,
        SUM(conversions) AS total_conversions
    FROM marts.fact_marketing
    GROUP BY channel
""").df()

fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(df['total_spend'], df['total_revenue'],
                     s=df['total_conversions'] / 5,
                     alpha=0.7, color='steelblue', edgecolors='white')

for _, row in df.iterrows():
    ax.annotate(row['channel'],
                xy=(row['total_spend'], row['total_revenue']),
                xytext=(8, 4), textcoords='offset points', fontsize=10)

ax.plot([0, df['total_spend'].max()], [0, df['total_spend'].max()],
        'r--', linewidth=0.8, label='Break-even line (ROAS = 1.0)')
ax.set_title('Spend vs revenue by channel\n(bubble size = total conversions)', fontsize=14, pad=12)
ax.set_xlabel('Total spend (USD)')
ax.set_ylabel('Total revenue attributed (USD)')
ax.legend()
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(
    output_dir / "spend_vs_revenue.png",
    dpi=150,
    bbox_inches='tight'
)
plt.show()

`Channels above the break-even line are profitable. Channels with small bubbles (low conversions) relative to their spend position have a unit economics problem - more spend is not the answer, conversion rate optimisation is.`

### Chart - CPA by channel (sorted ascending)

In [ ]:
df = con.execute("""
    SELECT channel, ROUND(AVG(cpa), 2) AS avg_cpa
    FROM marts.fact_marketing
    GROUP BY channel
    ORDER BY avg_cpa ASC
""").df()

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2ecc71' if i < 2 else '#e74c3c' if i > 2 else '#f39c12'
          for i in range(len(df))]
bars = ax.barh(df['channel'], df['avg_cpa'], color=colors)
ax.bar_label(bars, fmt='$%.2f', padding=4, fontsize=10)
ax.set_title('Average cost per acquisition by channel', fontsize=14, pad=12)
ax.set_xlabel('CPA (USD)')
plt.tight_layout()
output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(
    output_dir / "cpa.png",
    dpi=150,
    bbox_inches='tight'
)
plt.show()

`Email and Organic have the lowest CPA - reinforcing the priority to grow these channels before scaling paid. Reallocating 15–20% of high-CPA channel budget to Email nurture sequences could reduce blended CPA by a meaningful margin.`

### Chart - Best and worst 5 campaigns by ROAS

In [ ]:
df = con.execute("""
    SELECT
        channel || ' — ' || CAST(campaign_month AS VARCHAR) AS campaign_label,
        roas
    FROM marts.fact_marketing
    ORDER BY roas DESC
""").df()

top5 = df.head(5)
bottom5 = df.tail(5)
combined = pd.concat([top5, bottom5])
colors = ['#2ecc71'] * 5 + ['#e74c3c'] * 5

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(combined['campaign_label'], combined['roas'], color=colors)
ax.bar_label(bars, fmt='%.2f', padding=4, fontsize=10)
ax.axvline(1.0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Top 5 and bottom 5 campaigns by ROAS', fontsize=14, pad=12)
ax.set_xlabel('ROAS')
ax.invert_yaxis()
plt.tight_layout()
output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(
    output_dir / "campaign_roas.png",
    dpi=150,
    bbox_inches='tight'
)
plt.show()

`The gap between best and worst campaign ROAS within the same channel confirms that execution quality (timing, creative, targeting) drives results more than channel selection alone. The business should study what the top-performing campaigns did differently and build a replication checklist.`